# Chapter 05 데모 — 에이전틱 메모리: 다시 찾아온 고객을 기억하기

> 📑 대응 강의: **05-Memory.pptx** (에이전틱 메모리 구현)

| 챕터 | 노트북 | 고객지원 에이전트에 추가되는 기능 |
|---|---|---|
| 02 | `02_Runtime_Demo` | Strands 에이전트 → **AgentCore Runtime** 서버리스 배포, 세션 격리, 버전/엔드포인트 |
| 03 | `03_SecurityIdentity_Demo` | **Cognito JWT 인바운드 인증**, 사용자 식별·감사 로그, **API 키 아웃바운드 인증**(배송 파트너 API) |
| 04 | `04_ToolsGateway_Demo` | **Code Interpreter**, MCP, **AgentCore Gateway**(Lambda→MCP 도구), **AgentCore Policy**(Cedar) |
| 👉 **05** | `05_Memory_Demo` | **AgentCore Memory** 단기/장기 메모리 → 세션을 넘어 고객을 기억 |
| 06 | `06_Observability_Evaluation_Demo` | **Observability**(트레이스/스팬/지표), **Evaluations**(온디맨드/온라인), PROD 승격 |
| 99 | `99_Cleanup` | 데모 리소스 일괄 삭제 |

### 04장에서 남은 문제
- 세션(microVM)이 끝나면 대화 내용이 사라지고, **다음 방문 때 고객의 선호·이력을 모릅니다** (슬라이드 4, 7).

### 이번 챕터에서 할 일
| 단계 | 내용 | 슬라이드 |
|---|---|---|
| 1 | 현재 에이전트가 세션 간에 기억하지 못함을 확인 | 4 |
| 2 | **장기 메모리 전략**(사용자 선호 · 시맨틱 · 요약)과 **네임스페이스**를 가진 Memory 생성 | 13–15, 20, 24 |
| 3 | **단기 메모리**: 이벤트 저장/조회 (`create_event`, `list_events`) | 16–18, 23 |
| 4 | 단기 → 장기 메모리 **비동기 추출** 확인 (`retrieve_memories`) | 19, 22 |
| 5 | Strands **후크**로 메모리 검색·저장 자동화 | 25–28 |
| 6 | 런타임 에이전트(v4)에 결합 — **JWT 사용자 = 메모리 actor** | 16, 31 |
| 7 | 메모리 보안: actor/네임스페이스 단위 세분화된 접근 제어 | 29–32 |

```
 alice ─JWT─▶ Runtime(v4) ── Agent ──┬─ AgentCoreMemorySessionManager ──▶ AgentCore Memory
             (username=alice)        │      actor_id = alice                ├─ 단기: 이벤트(대화 원문)
                                     │      session_id = runtime 세션        └─ 장기: 선호 / 사실 / 세션 요약
                                     └─ 로컬 도구 · Code Interpreter · Gateway(Policy)
```

## 0. 환경 준비 & 이전 챕터 결과물 불러오기

In [ ]:
import json, time
import boto3
from IPython.display import Markdown, display

from demo_helpers import config
from demo_helpers.aws_setup import get_region, get_account_id, wait_for, get_bearer_token
from demo_helpers.invoke import invoke_with_jwt, new_session_id, show

REGION = get_region()
ACCOUNT_ID = get_account_id()
state = config.require_state("agent_arn", "agent_id", "runtime_role_arn", "gateway_url", "cognito_client_id")
agent_arn, agent_id = state["agent_arn"], state["agent_id"]
control = boto3.client("bedrock-agentcore-control", region_name=REGION)

alice_token = get_bearer_token("alice")
bob_token = get_bearer_token("bob")

## 1. 문제 확인: 현재 에이전트(v3)는 세션을 넘어 기억하지 못한다 (슬라이드 4)

In [ ]:
s1 = new_session_id("alice-visit1")
show(invoke_with_jwt(agent_arn, "저는 흰색 제품을 좋아하고 예산은 보통 40만원 이하예요. 지난달 AnyBook Pro 노트북도 여기서 샀어요.",
                     alice_token, s1))

In [ ]:
s2 = new_session_id("alice-visit2")   # 며칠 뒤 다시 방문
show(invoke_with_jwt(agent_arn, "제 취향에 맞는 헤드폰 추천해 주세요. 제가 전에 뭘 샀는지도 기억하시죠?", alice_token, s2))

## 2. AgentCore Memory 리소스 생성 (슬라이드 13–15, 20, 24)

| 메모리 유형 | 저장 내용 | AgentCore Memory 에서 |
|---|---|---|
| **단기 메모리** | 대화 원문, 세션 상태, 체크포인트 | **이벤트**(actor + session 단위) |
| **장기 메모리** | 처리된 정보: 선호, 사실, 요약 | **메모리 전략**이 이벤트에서 비동기로 추출한 **레코드** |

내장 전략 3가지를 사용하고, 고객(actor)별로 분리되도록 **네임스페이스**를 지정합니다.

| 전략 | 네임스페이스 |
|---|---|
| 사용자 선호 `userPreferenceMemoryStrategy` | `support/customer/{actorId}/preferences` |
| 시맨틱(사실) `semanticMemoryStrategy` | `support/customer/{actorId}/facts` |
| 요약 `summaryMemoryStrategy` | `support/customer/{actorId}/{sessionId}/summary` |

> **단기 메모리만** 필요하다면 전략 없이 생성합니다 (슬라이드 17):
> ```python
> client.create_memory(name="test_memory", description="Short-term memory only", event_expiry_days=7)
> ```

In [ ]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

memory_client = MemoryClient(region_name=REGION)

existing = [m for m in memory_client.list_memories()
            if m.get("id", "").startswith(config.MEMORY_NAME) and m.get("status") in ("ACTIVE", "CREATING")]
if existing:
    memory_id = existing[0]["id"]
    print("ℹ️  기존 Memory 재사용:", memory_id)
    wait_for(lambda: memory_client.get_memory_status(memory_id), ready={"ACTIVE"}, failed={"FAILED"}, label="Memory")
else:
    print("Memory 생성 중... (2~3분 소요)")
    memory = memory_client.create_memory_and_wait(
        name=config.MEMORY_NAME,
        description="Customer support agent memory",
        event_expiry_days=30,  # 단기 메모리(이벤트) 보존 기간
        strategies=[
            {StrategyType.USER_PREFERENCE.value: {
                "name": "CustomerPreferences",
                "description": "고객의 취향, 예산, 선호 브랜드/색상 등",
                "namespaces": [config.NS_PREFERENCES]}},
            {StrategyType.SEMANTIC.value: {
                "name": "CustomerFacts",
                "description": "구매 이력, 거주지 등 고객 관련 사실",
                "namespaces": [config.NS_FACTS]}},
            {StrategyType.SUMMARY.value: {
                "name": "SessionSummaries",
                "description": "상담 세션 요약",
                "namespaces": [config.NS_SUMMARY]}},
        ],
    )
    memory_id = memory["id"]
config.save_state(memory_id=memory_id)

for s in memory_client.get_memory_strategies(memory_id):
    print(f"- {s.get('type', ''):16} {s.get('name', ''):20} {s.get('namespaces')}")

## 3. 단기 메모리: 이벤트 저장 및 조회 (슬라이드 16–18, 23)

- **이벤트**: 사용자·에이전트·도구 메시지 묶음 (1개 또는 여러 개를 한 번에 저장)
- **행위자(actor)**: 메모리의 주인 — 여기서는 로그인한 고객 ID
- **세션**: 하나의 대화 단위

In [ ]:
ACTOR = "alice"
STM_SESSION = new_session_id("alice-stm-demo")

conversation = [
    [("안녕하세요, 저는 흰색 제품을 좋아하고 예산은 보통 40만원 이하예요.", "USER"),
     ("흰색 · 40만원 이하 선호를 참고하겠습니다. 무엇을 도와드릴까요?", "ASSISTANT")],
    [("지난달 AnyBook Pro 14 노트북(시리얼 SN1000001)을 샀는데 보증이 언제까지죠?", "USER"),
     ("check_warranty_status(serial_number='SN1000001')", "TOOL"),
     ("AnyBook Pro 14 의 보증은 2027-05-31 까지 유효합니다.", "ASSISTANT")],
    [("그리고 저는 서울 강남구에 살아요. 주말에만 택배를 받을 수 있어요.", "USER"),
     ("알겠습니다. 주말 배송 선호를 기록해 두겠습니다.", "ASSISTANT")],
]
for messages in conversation:
    event = memory_client.create_event(memory_id=memory_id, actor_id=ACTOR, session_id=STM_SESSION, messages=messages)
    print("📝 이벤트 저장:", event["eventId"])

In [ ]:
events = memory_client.list_events(memory_id=memory_id, actor_id=ACTOR, session_id=STM_SESSION)
print(f"이벤트 {len(events)}개")
for ev in events:
    for p in ev.get("payload", []):
        conv = p.get("conversational", {})
        print(f"  [{conv.get('role')}] {conv.get('content', {}).get('text')}")

In [ ]:
# 최근 K 턴만 가져오기 — 에이전트 컨텍스트 복원에 사용
for turn in memory_client.get_last_k_turns(memory_id=memory_id, actor_id=ACTOR, session_id=STM_SESSION, k=2):
    print(" | ".join(f"{m['role']}: {m['content']['text'][:40]}" for m in turn))

## 4. 단기 → 장기 메모리 추출 (슬라이드 19, 22)

이벤트가 저장되면 메모리 전략이 **비동기로** 실행되어 정보를 추출하고, 비슷한 기존 레코드와 비교해 **추가/갱신/생략(ADD/UPDATE/SKIP)** 합니다.
보통 1분 내외가 걸립니다.

In [ ]:
def show_long_term(actor, session_id=None, query="고객 정보"):
    namespaces = {"선호": config.NS_PREFERENCES, "사실": config.NS_FACTS}
    if session_id:
        namespaces["요약"] = config.NS_SUMMARY
    total = 0
    for label, ns in namespaces.items():
        ns = ns.format(actorId=actor, sessionId=session_id)
        records = memory_client.retrieve_memories(memory_id=memory_id, namespace=ns, query=query, top_k=5)
        total += len(records)
        print(f"── [{label}] {ns}")
        for r in records:
            print("   •", r["content"]["text"][:200])
    return total

for i in range(12):
    if show_long_term(ACTOR) >= 2:
        break
    print(f"⏳ 장기 메모리 추출 대기 중... ({(i + 1) * 15}s)\n")
    time.sleep(15)

In [ ]:
# 세션 요약 (summary 네임스페이스는 세션 단위)
show_long_term(ACTOR, session_id=STM_SESSION, query="상담 요약")

## 5. Strands 후크로 메모리 자동화 (슬라이드 27–28)

- `MessageAddedEvent` (사용자 메시지 추가 시) → 장기 메모리 **검색**해 컨텍스트로 주입
- `AfterInvocationEvent` (응답 완료 후) → 이번 턴을 단기 메모리에 **저장**

> 도구 방식(슬라이드 25–26): `strands_tools.agent_core_memory.AgentCoreMemoryToolProvider` 를 쓰면
> 에이전트가 스스로 판단해 메모리를 저장/검색하는 **도구**로 사용할 수도 있습니다.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.hooks import AfterInvocationEvent, HookProvider, HookRegistry, MessageAddedEvent
from demo_helpers.tools import SYSTEM_PROMPT, get_product_info, get_return_policy


def _text(message):
    return "".join(c.get("text", "") for c in message["content"])


class CustomerMemoryHooks(HookProvider):
    def __init__(self, memory_id, actor_id, session_id):
        self.memory_id, self.actor_id, self.session_id = memory_id, actor_id, session_id
        self.last_query = None

    def retrieve_context(self, event: MessageAddedEvent):
        message = event.message
        if message["role"] != "user" or any("toolResult" in c for c in message["content"]):
            return
        self.last_query = _text(message)
        context = []
        for label, ns in (("선호", config.NS_PREFERENCES), ("사실", config.NS_FACTS)):
            for r in memory_client.retrieve_memories(memory_id=self.memory_id, namespace=ns.format(actorId=self.actor_id),
                                                     query=self.last_query, top_k=3):
                context.append(f"[{label}] {r['content']['text']}")
        if context:
            print("🧠 주입된 장기 메모리:\n   " + "\n   ".join(context))
            message["content"][0]["text"] = "<고객 기억 정보>\n" + "\n".join(context) + "\n</고객 기억 정보>\n\n" + self.last_query

    def save_turn(self, event: AfterInvocationEvent):
        answer = _text(event.agent.messages[-1])
        if self.last_query and answer:
            memory_client.create_event(memory_id=self.memory_id, actor_id=self.actor_id, session_id=self.session_id,
                                       messages=[(self.last_query, "USER"), (answer, "ASSISTANT")])

    def register_hooks(self, registry: HookRegistry, **kwargs):
        registry.add_callback(MessageAddedEvent, self.retrieve_context)
        registry.add_callback(AfterInvocationEvent, self.save_turn)


hook_agent = Agent(
    model=BedrockModel(model_id=config.MODEL_ID, region_name=REGION, temperature=0.3),
    tools=[get_product_info, get_return_policy],
    system_prompt=SYSTEM_PROMPT,
    hooks=[CustomerMemoryHooks(memory_id, actor_id="alice", session_id=new_session_id("alice-hooks"))],
)
hook_agent("헤드폰 하나 추천해 주실래요?")

👉 새 세션인데도 *흰색 · 40만원 이하* 선호를 반영해 추천합니다.

## 6. 런타임 에이전트(v4)에 결합

`# [CH05]` 표시 부분이 바뀝니다.
- 04장의 microVM 메모리(`_conversation`) 대신 **`AgentCoreMemorySessionManager`** 사용
  - 단기 메모리: 런타임 세션 ID 로 대화를 저장/복원 → microVM 이 교체되어도 대화 유지
  - 장기 메모리: `retrieval_config` 의 네임스페이스에서 관련 기억을 자동 검색해 주입
- **actor_id = JWT 의 username** → 03장의 인증이 곧 메모리 격리의 기준이 됩니다.

In [ ]:
%%writefile agent/runtime_ch05.py
"""[Chapter 05] + AgentCore Memory (단기: 세션 대화 / 장기: 선호·사실·요약), actor = 인증된 고객."""
import base64
import hashlib
import json
import os
import threading
import time
import uuid

from strands import Agent, tool
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp.client.streamable_http import streamablehttp_client
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from bedrock_agentcore.identity.auth import requires_api_key
from bedrock_agentcore.tools.code_interpreter_client import code_session
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig  # [CH05]
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager  # [CH05]

from demo_helpers.config import MODEL_ID, PARTNER_API_PROVIDER, NS_FACTS, NS_PREFERENCES
from demo_helpers.tools import SYSTEM_PROMPT, get_product_info, get_return_policy

REGION = os.environ.get("AWS_REGION", "us-east-1")
GATEWAY_URL = os.environ["GATEWAY_URL"]
MEMORY_ID = os.environ["MEMORY_ID"]  # [CH05]
MICROVM_BOOT_ID = uuid.uuid4().hex[:8]

app = BedrockAgentCoreApp()


@tool
def generate_sales_report(duration_seconds: int = 20) -> str:
    """장시간 걸리는 '판매 리포트 생성' 작업을 백그라운드로 시작하고 즉시 응답합니다.

    Args:
        duration_seconds: 작업 소요 시간(초)
    """
    task_id = app.add_async_task("sales_report", {"duration": duration_seconds})

    def background_work():
        time.sleep(duration_seconds)
        app.complete_async_task(task_id)

    threading.Thread(target=background_work, daemon=True).start()
    return f"리포트 작업(ID: {task_id})을 시작했습니다. 약 {duration_seconds}초 후 완료됩니다."


@requires_api_key(provider_name=PARTNER_API_PROVIDER)
def _get_partner_api_key(*, api_key: str = "") -> str:
    return api_key


@tool
def track_shipment(order_id: str) -> str:
    """주문 번호로 배송 상태를 조회합니다 (외부 배송 파트너 API 호출).

    Args:
        order_id: 주문 번호 (예: ORD-1001)
    """
    api_key = _get_partner_api_key()
    statuses = ["상품 준비 중", "배송 중 (서울 허브 도착)", "배송 완료"]
    status = statuses[int(hashlib.md5(order_id.encode()).hexdigest(), 16) % len(statuses)]
    return f"주문 {order_id}: {status} (파트너 API 키 ****{api_key[-4:]} 로 인증됨)"


@tool
def run_python(code: str) -> str:
    """금액 계산·데이터 분석이 필요할 때 Python 코드를 안전한 샌드박스에서 실행합니다. 결과는 print() 로 출력하세요.

    Args:
        code: 실행할 Python 코드
    """
    with code_session(REGION) as ci:
        response = ci.invoke("executeCode", {"code": code, "language": "python", "clearContext": True})
        for event in response["stream"]:
            return json.dumps(event["result"].get("structuredContent", event["result"]), ensure_ascii=False)


def get_caller_claims(context) -> dict:
    auth = (context.request_headers or {}).get("Authorization", "")
    if not auth.startswith("Bearer "):
        return {}
    payload = auth.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


_session_owner = None


@app.entrypoint
def invoke(payload, context):
    global _session_owner
    claims = get_caller_claims(context)
    username = claims.get("username", "anonymous")
    if _session_owner and _session_owner != username:
        print(json.dumps({"type": "AUDIT", "event": "SESSION_OWNER_MISMATCH", "user": username,
                          "session_id": context.session_id}))
        return {"response": "⛔ 이 세션은 다른 사용자에게 속해 있어 사용할 수 없습니다.", "user": username}
    _session_owner = username

    # [CH05] 인증된 고객 = 메모리 actor, 런타임 세션 = 메모리 세션
    memory_config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        actor_id=username,
        session_id=context.session_id,
        retrieval_config={
            NS_PREFERENCES: RetrievalConfig(top_k=3, relevance_score=0.2),
            NS_FACTS: RetrievalConfig(top_k=5, relevance_score=0.2),
        },
    )

    auth_header = context.request_headers["Authorization"]
    gateway = MCPClient(lambda: streamablehttp_client(GATEWAY_URL, headers={"Authorization": auth_header}))

    prompt = payload.get("prompt", "")
    with gateway:
        gateway_tools = gateway.list_tools_sync()
        agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.3),
            tools=[get_product_info, get_return_policy, generate_sales_report, track_shipment, run_python]
            + gateway_tools,
            system_prompt=SYSTEM_PROMPT
            + f"\n현재 로그인한 고객 ID: {username}\n"
            + "환불 요청이 정책에 의해 거부되면 고객에게 상담원 연결(1588-0000)을 안내하세요.\n"
            + "고객에 대한 기억 정보가 제공되면 이를 활용해 개인화된 답변을 하세요.\n",
            session_manager=AgentCoreMemorySessionManager(memory_config, REGION),  # [CH05]
            callback_handler=None,
        )
        result = agent(prompt)
    tools_used = list(result.metrics.tool_metrics.keys())

    print(json.dumps({
        "type": "AUDIT", "user": username, "sub": claims.get("sub"),
        "session_id": context.session_id, "prompt": prompt[:200], "tools_used": tools_used,
    }, ensure_ascii=False))

    return {
        "response": str(result),
        "user": username,
        "session_id": context.session_id,
        "microvm_boot_id": MICROVM_BOOT_ID,
        "tools_used": tools_used,
        "messages_in_session": len(agent.messages),
    }


if __name__ == "__main__":
    app.run()

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

state = config.load_state()
agentcore_runtime = Runtime()
agentcore_runtime.configure(
    entrypoint="agent/runtime_ch05.py",
    agent_name=config.AGENT_NAME,
    execution_role=state["runtime_role_arn"],
    auto_create_ecr=True,
    requirements_file="agent/requirements.txt",
    region=REGION,
    authorizer_configuration={"customJWTAuthorizer": {
        "discoveryUrl": state["cognito_discovery_url"], "allowedClients": [state["cognito_client_id"]]}},
    request_header_configuration={"requestHeaderAllowlist": ["Authorization"]},
)
agentcore_runtime.launch(auto_update_on_conflict=True,
                         env_vars={"GATEWAY_URL": state["gateway_url"], "MEMORY_ID": memory_id})
wait_for(lambda: agentcore_runtime.status().endpoint["status"],
         ready={"READY"}, failed={"CREATE_FAILED", "UPDATE_FAILED"}, interval=10, label="Runtime DEFAULT 엔드포인트")
print("최신 버전:", control.get_agent_runtime(agentRuntimeId=agent_id)["agentRuntimeVersion"])

### 6-1. 1단계와 같은 질문을 다시 — 이번에는 기억합니다

In [ ]:
show(invoke_with_jwt(agent_arn, "제 취향에 맞는 헤드폰 추천해 주세요. 제가 전에 뭘 샀는지도 기억하시죠?",
                     alice_token, new_session_id("alice-visit3")))

### 6-2. 새로운 사실을 알려주면 → 장기 메모리로 추출 → 다음 방문에 활용

In [ ]:
visit4 = new_session_id("alice-visit4")
show(invoke_with_jwt(agent_arn, "참, 저 다음 달에 부산 해운대로 이사 가요. 그리고 모니터도 하나 살까 생각 중이에요.",
                     alice_token, visit4))

In [ ]:
# 단기 메모리: 런타임 세션 = 메모리 세션 으로 대화가 저장되었는지 확인
for ev in memory_client.list_events(memory_id=memory_id, actor_id="alice", session_id=visit4):
    for p in ev.get("payload", []):
        conv = p.get("conversational", {})
        if conv:
            print(f"[{conv['role']}] {conv['content']['text'][:120]}")

In [ ]:
# 장기 메모리 추출 대기 (부산 이사 사실이 기록될 때까지)
for i in range(12):
    facts = memory_client.retrieve_memories(memory_id=memory_id, namespace=config.NS_FACTS.format(actorId="alice"),
                                            query="이사 거주지 부산", top_k=5)
    if any("부산" in r["content"]["text"] for r in facts):
        for r in facts:
            print("•", r["content"]["text"])
        break
    print(f"⏳ 추출 대기 중... ({(i + 1) * 15}s)")
    time.sleep(15)

In [ ]:
show(invoke_with_jwt(agent_arn, "이사 가는 동네 근처에 서비스센터 있나요? 그리고 저한테 맞는 모니터 추천해 주세요.",
                     alice_token, new_session_id("alice-visit5")))

### 6-3. 고객 간 메모리 격리 — bob 은 alice 의 기억에 접근할 수 없음

In [ ]:
show(invoke_with_jwt(agent_arn, "제가 어디로 이사 간다고 했죠? 제 예산이 얼마였죠?", bob_token, new_session_id("bob")))

## 7. 메모리 보안: 세분화된 접근 제어 (슬라이드 29–32)

- Memory 데이터는 기본적으로 **암호화**(AWS 관리형 키 또는 고객 관리형 KMS 키)되어 저장됩니다.
- 이 데모는 에이전트 코드가 actor 를 JWT 에서 결정하도록 해 격리했습니다.
- 더 강하게 격리하려면 IAM 조건 키로 **actor / 네임스페이스 단위 접근**을 제한합니다. 예: *alice 의 선호 네임스페이스만 검색 허용*

In [ ]:
policy_example = {
    "Version": "2012-10-17",
    "Statement": [{
        "Sid": "AliceNamespaceOnly",
        "Effect": "Allow",
        "Action": ["bedrock-agentcore:RetrieveMemoryRecords"],
        "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:memory/{memory_id}",
        "Condition": {"StringEquals": {"bedrock-agentcore:namespace": config.NS_PREFERENCES.format(actorId="alice")}},
    }, {
        "Sid": "AliceEventsOnly",
        "Effect": "Allow",
        "Action": ["bedrock-agentcore:CreateEvent", "bedrock-agentcore:ListEvents"],
        "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{ACCOUNT_ID}:memory/{memory_id}",
        "Condition": {"StringEquals": {"bedrock-agentcore:actorId": "alice"}},
    }],
}
print(json.dumps(policy_example, indent=2, ensure_ascii=False))

## ✅ Chapter 05 정리

| 달성한 것 | 아직 부족한 것 (→ 다음 챕터) |
|---|---|
| 단기 메모리로 세션 대화 저장·복원 (microVM 교체에도 유지) | 에이전트 내부에서 **무슨 일이 일어났는지 보이지 않음** (어떤 도구를 몇 번, 얼마나 오래?) |
| 선호·사실·요약을 장기 메모리로 비동기 추출, 다음 방문에 개인화 | 응답 **품질을 객관적으로 측정**할 방법 없음 |
| 인증된 사용자 = 메모리 actor → 고객 간 기억 격리 | `PROD` 엔드포인트는 여전히 V1 |

➡️ **[06_Observability_Evaluation_Demo](06_Observability_Evaluation_Demo.ipynb)**: 지금까지 만든 에이전트를 관찰하고 평가한 뒤 프로덕션으로 승격합니다.